## 1. Creación del entorno

#### 1.1 Creamos nuestra estructura dentro de Unity Catalog

In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS sesion1")
spark.sql("CREATE SCHEMA IF NOT EXISTS sesion1.data")
spark.sql("CREATE VOLUME IF NOT EXISTS sesion1.data.landing")

DataFrame[]

#### 1.2 Incorporación de datos a Volumenes

In [0]:
%sh
curl -L https://raw.githubusercontent.com/regarcia-magister/EAM-ETL-IA/refs/heads/main/sesion%201-2/data/nutrients_csvfile.csv -o /Volumes/sesion1/data/landing/nutrients_csvfile.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 19760  100 19760    0     0   4439      0  0:00:04  0:00:04 --:--:--  4439


#### 1.3 Crear una tabla

In [0]:
%sql
CREATE TABLE IF NOT EXISTS sesion1.data.tabla_simple (
  id INT,
  letra STRING,
  valor DOUBLE
);

INSERT INTO sesion1.data.tabla_simple
VALUES (1, 'A', 10.5),
       (2, 'B', 20.0),
       (3, 'C', 30.75);

num_affected_rows,num_inserted_rows
3,3


#### 1.4 Crear una vista

In [0]:
CREATE OR REPLACE VIEW sesion1.data.vw_tabla_simple AS
SELECT id, letra
FROM sesion1.data.tabla_simple
WHERE valor > 15;


#### 1.5 Crear una función

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS sesion1.security")

DataFrame[]

In [0]:
%sql

CREATE OR REPLACE FUNCTION sesion1.security.fn_mayor_que_10(x DOUBLE)
RETURNS BOOLEAN
RETURN x > 10

In [0]:
%sql
-- Verificamos si la función funciona
SELECT * FROM sesion1.data.tabla_simple WHERE sesion1.security.fn_mayor_que_10(valor);


id,letra,valor
1,A,10.5
2,B,20.0
3,C,30.75


#### 1.6 Crear un Dataframe

In [0]:
path_data_demo = "/Volumes/sesion1/data/landing/data_demo.csv"
df = spark.read.csv(path_data_demo, header=True, inferSchema=True)
display(df)

rooms,garages,useful_area,value,interior_quality,time_on_market,has_outlier
3.0,1.0,105.0,1038640.0,2,31.7870789438591,0
3.0,2.0,76.0,606405.0,4,209.46884172046086,0
3.0,1.0,123.0,1534500.0,1,38.0,0
3.0,2.0,180.0,1131950.0,5,154.0,0
3.0,1.0,67.0,452672.0,4,15.0,0
3.0,3.0,198.0,3116290.0,3,306.3557294709182,0
4.0,2.0,245.0,1202560.0,3,24.824464122611424,0
2.0,2.0,66.0,419282.0,5,24.0,0
2.0,1.0,114.0,1830260.0,1,251.39533835163672,0
2.0,1.0,56.0,274005.0,5,93.0,0


In [0]:
%sh
curl -L https://raw.githubusercontent.com/regarcia-magister/EAM-ETL-IA/refs/heads/main/sesion%201-2/data/ventas_2025.csv -o /Volumes/sesion1/data/landing/ventas_2025.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  3072  100  3072    0     0    996      0  0:00:03  0:00:03 --:--:--   996


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

schema = StructType([
    StructField("ID", StringType(), True),
    StructField("fecha", StringType(), True),
    StructField("producto", StringType(), True),
    StructField("cantidad", IntegerType(), True),
    StructField("precio", DoubleType(), True)
])

df_csv = spark.read.csv(
    "/Volumes/sesion1/data/landing/ventas_2025.csv",
    header=True,
    schema=schema,
    sep=","
)

In [0]:
display(df_csv)

ID,fecha,producto,cantidad,precio
1,2025-01-03,Teclado,20,25.99
2,2025-01-03,Webcam,1,39.99
3,2025-01-03,Auriculares,18,29.99
4,2025-01-04,Webcam,10,39.99
5,2025-01-04,Mouse,2,15.5
6,2025-01-05,Altavoces,13,45.0
7,2025-01-08,Teclado,11,25.99
8,2025-01-10,Auriculares,5,29.99
9,2025-01-11,Monitor,2,199.99
10,2025-01-12,Microfono,20,59.99


In [0]:
path_data_ventas = "/Volumes/sesion1/data/landing/ventas_2025.csv"
df3 = spark.read.csv(path_data_ventas, header=True, inferSchema=True)
display(df3)

id,fecha,producto,cantidad,precio
1,2025-01-03,Teclado,20,25.99
2,2025-01-03,Webcam,1,39.99
3,2025-01-03,Auriculares,18,29.99
4,2025-01-04,Webcam,10,39.99
5,2025-01-04,Mouse,2,15.5
6,2025-01-05,Altavoces,13,45.0
7,2025-01-08,Teclado,11,25.99
8,2025-01-10,Auriculares,5,29.99
9,2025-01-11,Monitor,2,199.99
10,2025-01-12,Microfono,20,59.99


In [0]:
path_data_demo2 = "/Volumes/sesion1/data/landing/dim_spark_intro.csv"
df2 = spark.read.csv(path_data_demo2, header=True, inferSchema=True)
display(df2)

product,product_family,supplier
Laptop,Computers,TechCorp
Mouse,Peripherals,InputMakers
Keyboard,Peripherals,InputMakers
Monitor,Displays,ScreenWorld


#### 1.7 Guardar en Delta lake

In [0]:
df_csv.write.format("delta").mode("overwrite").saveAsTable("sesion1.data.productos")


In [0]:
%sql
SELECT * FROM sesion1.data.productos;

ID,fecha,producto,cantidad,precio
1,2025-01-03,Teclado,20,25.99
2,2025-01-03,Webcam,1,39.99
3,2025-01-03,Auriculares,18,29.99
4,2025-01-04,Webcam,10,39.99
5,2025-01-04,Mouse,2,15.5
6,2025-01-05,Altavoces,13,45.0
7,2025-01-08,Teclado,11,25.99
8,2025-01-10,Auriculares,5,29.99
9,2025-01-11,Monitor,2,199.99
10,2025-01-12,Microfono,20,59.99


## 2. Modificar una tabla

#### 2.1 INSERT

In [0]:
%sql
INSERT INTO sesion1.data.productos (id, fecha, producto, cantidad, precio)
VALUES ("A001", "2025-11-25", "Torre E-138", 3, 1200000.50)

num_affected_rows,num_inserted_rows
1,1


#### 2.2 UPDATE

In [0]:
%sql

UPDATE sesion1.data.productos
SET cantidad = 5, 
    precio = 119000.00
WHERE id = "1";

num_affected_rows
1


In [0]:
%sql
SELECT * FROM sesion1.data.productos;

ID,fecha,producto,cantidad,precio
2,2025-01-03,Webcam,1,39.99
3,2025-01-03,Auriculares,18,29.99
4,2025-01-04,Webcam,10,39.99
5,2025-01-04,Mouse,2,15.5
6,2025-01-05,Altavoces,13,45.0
7,2025-01-08,Teclado,11,25.99
8,2025-01-10,Auriculares,5,29.99
9,2025-01-11,Monitor,2,199.99
10,2025-01-12,Microfono,20,59.99
11,2025-01-12,USB,8,9.99


#### 2.3 MERGE

In [0]:
%sql

MERGE 

#### 2.3 Time Travel

In [0]:
%sql
DESCRIBE HISTORY sesion1.data.productos

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-03-14T19:27:25.000Z,73307969058957,vdiegoj25@gmail.com,UPDATE,"Map(predicate -> [""(id#14157 = 1)""])",null,List(4138575896365003),b24a2dd4-758d-40d3-a35a-f6d6d15fb360,0314-190451-kgsig3oq-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3907, numDeletionVectorsUpdated -> 0, scanTimeMs -> 1985, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1453, rewriteTimeMs -> 1887)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
1,2026-03-14T19:24:03.000Z,73307969058957,vdiegoj25@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(4138575896365003),98223140-8881-4746-8f84-8038d6ba776d,0314-190451-kgsig3oq-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1459)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
0,2026-03-14T19:13:31.000Z,73307969058957,vdiegoj25@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(4138575896365003),ac085c16-bfac-4a1d-9b74-6ef0ff6a5176,0314-190451-kgsig3oq-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 100, numOutputBytes -> 2318)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13


In [0]:
%sql
SELECT * FROM sesion1.data.productos VERSION AS OF 1;

ID,fecha,producto,cantidad,precio
1,2025-01-03,Teclado,20,25.99
2,2025-01-03,Webcam,1,39.99
3,2025-01-03,Auriculares,18,29.99
4,2025-01-04,Webcam,10,39.99
5,2025-01-04,Mouse,2,15.5
6,2025-01-05,Altavoces,13,45.0
7,2025-01-08,Teclado,11,25.99
8,2025-01-10,Auriculares,5,29.99
9,2025-01-11,Monitor,2,199.99
10,2025-01-12,Microfono,20,59.99


In [0]:
%sql
CREATE TABLE sesion1.data.productos_V3
AS SELECT * FROM sesion1.data.productos VERSION AS OF 2;


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM sesion1.data.productos_V3

#### 2.4 Select(): elegir columnas

In [0]:
sales_simple_df = sales_df.select("order_id", "order_date", "country", "units_sold",)
display(sales_simple_df)

order_id,order_date,country,units_sold
1,2024-01-02,Spain,2
2,2024-01-03,Spain,5
3,2024-01-05,France,1
4,2024-01-06,Germany,3
5,2024-01-07,Spain,2
6,2024-01-08,France,10
7,2024-01-10,Germany,1
8,2024-01-11,Spain,4
9,2024-01-12,France,1
10,2024-01-14,Germany,6


## 3. Introducción a PySpark

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS sesion1.sparkintro")
spark.sql("CREATE VOLUME IF NOT EXISTS sesion1.sparkintro.landing")

DataFrame[]

In [0]:
%sh
curl -L https://raw.githubusercontent.com/regarcia-magister/EAM-ETL-IA/refs/heads/main/sesion%201-2/data/spark_intro.csv -o /Volumes/sesion1/sparkintro/landing/spark_intro.csv

curl -L https://raw.githubusercontent.com/regarcia-magister/EAM-ETL-IA/refs/heads/main/sesion%201-2/data/dim_spark_intro.csv -o /Volumes/sesion1/sparkintro/landing/dim_spark_intro.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   533  100   533    0     0    142      0  0:00:03  0:00:03 --:--:--   142
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   158  100   158    0     0   2084      0 --:--:-- --:--:-- --:--:--  2106


In [0]:
sales_df = (spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv("/Volumes/sesion1/sparkintro/landing/spark_intro.csv"))

display(sales_df)

order_id,order_date,country,product,category,units_sold,unit_price
1,2024-01-02,Spain,Laptop,Electronics,2,1200
2,2024-01-03,Spain,Mouse,Accessories,5,25
3,2024-01-05,France,Laptop,Electronics,1,1150
4,2024-01-06,Germany,Keyboard,Accessories,3,45
5,2024-01-07,Spain,Monitor,Electronics,2,300
6,2024-01-08,France,Mouse,Accessories,10,23
7,2024-01-10,Germany,Laptop,Electronics,1,1300
8,2024-01-11,Spain,Keyboard,Accessories,4,42
9,2024-01-12,France,Monitor,Electronics,1,280
10,2024-01-14,Germany,Mouse,Accessories,6,27


In [0]:
from pyspark.sql import functions as F

In [0]:
sales_with_total_df = sales_df.withColumn("total_sales", F.col("units_sold") * F.col("unit_price"))
display(sales_with_total_df)

order_id,order_date,country,product,category,units_sold,unit_price,total_sales
1,2024-01-02,Spain,Laptop,Electronics,2,1200,2400
2,2024-01-03,Spain,Mouse,Accessories,5,25,125
3,2024-01-05,France,Laptop,Electronics,1,1150,1150
4,2024-01-06,Germany,Keyboard,Accessories,3,45,135
5,2024-01-07,Spain,Monitor,Electronics,2,300,600
6,2024-01-08,France,Mouse,Accessories,10,23,230
7,2024-01-10,Germany,Laptop,Electronics,1,1300,1300
8,2024-01-11,Spain,Keyboard,Accessories,4,42,168
9,2024-01-12,France,Monitor,Electronics,1,280,280
10,2024-01-14,Germany,Mouse,Accessories,6,27,162


#### 2.5 Groupby().agg(): agregaciones

In [0]:
sales_country_df = sales_with_total_df.groupBy("country").agg(F.sum("units_sold").alias("total_units_sold"),
                                                              F.sum("total_sales").alias("total_sales"))
display(sales_country_df)

country,total_units_sold,total_sales
Spain,13,3293
France,12,1660
Germany,10,1597


#### Join()

In [0]:
product_dim_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/sesion1/sparkintro/landing/dim_spark_intro.csv")
)
display(product_dim_df)

product,product_family,supplier
Laptop,Computers,TechCorp
Mouse,Peripherals,InputMakers
Keyboard,Peripherals,InputMakers
Monitor,Displays,ScreenWorld


In [0]:
sales_completo_df = sales_with_total_df.alias("s").join(product_dim_df.alias("p"), on="product", how="left")
display(sales_completo_df)

product,order_id,order_date,country,category,units_sold,unit_price,total_sales,product_family,supplier
Laptop,1,2024-01-02,Spain,Electronics,2,1200,2400,Computers,TechCorp
Mouse,2,2024-01-03,Spain,Accessories,5,25,125,Peripherals,InputMakers
Laptop,3,2024-01-05,France,Electronics,1,1150,1150,Computers,TechCorp
Keyboard,4,2024-01-06,Germany,Accessories,3,45,135,Peripherals,InputMakers
Monitor,5,2024-01-07,Spain,Electronics,2,300,600,Displays,ScreenWorld
Mouse,6,2024-01-08,France,Accessories,10,23,230,Peripherals,InputMakers
Laptop,7,2024-01-10,Germany,Electronics,1,1300,1300,Computers,TechCorp
Keyboard,8,2024-01-11,Spain,Accessories,4,42,168,Peripherals,InputMakers
Monitor,9,2024-01-12,France,Electronics,1,280,280,Displays,ScreenWorld
Mouse,10,2024-01-14,Germany,Accessories,6,27,162,Peripherals,InputMakers


In [0]:
sales_completo_df.write.format("delta").mode("overwrite").saveAsTable("sesion1.sparkintro.sales_completo")